# Solutions: Panel Unit Root Tests

Solutions for Exercises 1-3 from Notebook 01.

**Note:** Exercise 4 is independent work -- no solution provided.

In [ ]:
%matplotlib inline

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

import sys

import panelbox
from panelbox.validation.unit_root import IPSTest, LLCTest

BASE_DIR = Path("..")
sys.path.insert(0, str(BASE_DIR / "utils"))
from unit_root_helpers import compare_unit_root_tests, interpret_results, recommend_transformation

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
np.random.seed(42)

DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "unit_root"
TABLES_DIR = BASE_DIR / "outputs" / "tables" / "unit_root"

for d in [FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PanelBox version: {panelbox.__version__}")
print("Setup complete!")

---

## Exercise 1: Basic Unit Root Test (Easy)

**Task:** Test whether log(prices) has a unit root using the regional prices panel dataset.

**Expected result:** Log prices should be I(1); inflation should be I(0).

In [ ]:
# Step 1: Load the data
prices = pd.read_csv(DATA_DIR / "unit_root" / "prices_panel.csv")

print("Prices Panel Dataset")
print("=" * 60)
print(f"Shape: {prices.shape}")
print(f"Regions: {prices['region'].nunique()}")
print(f"Years: {prices['year'].min()} - {prices['year'].max()}")
print(f"\nColumns: {list(prices.columns)}")
print("\nFirst rows:")
display(prices.head(10))
print("\nSummary:")
display(prices.describe())

In [ ]:
# Step 2: Run IPS test on log_price with trend='ct'
print("IPS Test on log(price)")
print("=" * 60)
try:
    ips = IPSTest(prices, "log_price", "region", "year", trend="ct")
    ips_result = ips.run()
    print("H0: All panels contain unit roots")
    print("H1: Some panels are stationary")
    print(f"\nW-statistic: {ips_result.statistic:.4f}")
    print(f"t-bar: {ips_result.t_bar:.4f}")
    print(f"P-value: {ips_result.pvalue:.4f}")
    print(f"\nConclusion: {ips_result.conclusion}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Step 3: Interpret -- run all tests for a comprehensive view
print("Comprehensive Unit Root Tests on log(price)")
print("=" * 60)
price_comparison = compare_unit_root_tests(
    prices,
    "log_price",
    "region",
    "year",
    trend="ct",
    save_path=str(TABLES_DIR / "01_ex1_price_unit_root.csv"),
)
display(price_comparison)

# Build interpretation dict
price_results_dict = {}
for _, row in price_comparison.iterrows():
    test_name = row["Test"].split("(")[0].strip()
    h0_type = "stationarity" if "Hadri" in row["Test"] else "unit_root"
    if not np.isnan(row["P-value"]):
        price_results_dict[test_name] = {"pvalue": row["P-value"], "h0": h0_type}

print("\n" + interpret_results(price_results_dict))
print(f"\nRecommendation: {recommend_transformation(price_results_dict)}")

In [ ]:
# Step 4: Test first difference (inflation) -- should be I(0)
# The 'inflation' column is already the first difference of log_price
# Drop first observation per region (inflation = 0 at t=0)
prices_infl = prices[prices["inflation"] != 0].copy()

print("Unit Root Tests on Inflation (first difference of log_price)")
print("=" * 60)
infl_comparison = compare_unit_root_tests(
    prices_infl,
    "inflation",
    "region",
    "year",
    trend="c",
    save_path=str(TABLES_DIR / "01_ex1_inflation_unit_root.csv"),
)
display(infl_comparison)

infl_results_dict = {}
for _, row in infl_comparison.iterrows():
    test_name = row["Test"].split("(")[0].strip()
    h0_type = "stationarity" if "Hadri" in row["Test"] else "unit_root"
    if not np.isnan(row["P-value"]):
        infl_results_dict[test_name] = {"pvalue": row["P-value"], "h0": h0_type}

print(f"\nRecommendation: {recommend_transformation(infl_results_dict)}")
print("\nConclusion: log(price) is I(1), inflation is I(0) -- as expected.")
print("Price indices drift over time (inflation), but inflation rates are stationary.")

### Exercise 1 Discussion

**Results:**
- **log(price)**: All tests with H0: unit root fail to reject; Hadri rejects stationarity. This confirms log prices are **I(1)**.
- **inflation**: Tests with H0: unit root reject; Hadri fails to reject stationarity. Inflation is **I(0)**.

**Economic interpretation:**
Price indices accumulate inflation over time, creating a non-stationary trend. The inflation rate itself (the change in log prices) is stationary -- it fluctuates around a mean without drifting permanently. This is a classic example of an I(1) variable whose first difference is I(0).

---

## Exercise 2: Comparing Tests (Medium)

**Task:** Compare IPS and LLC on homogeneous (G7) vs. heterogeneous (all 30 countries) panels.

In [ ]:
# Load Penn World Table data
pwt = pd.read_csv(DATA_DIR / "unit_root" / "penn_world_table.csv")

# Panel 1: G7-like countries (homogeneous -- similar developed economies)
g7_codes = ["AUS", "CAN", "DEU", "FRA", "GBR", "ITA", "JPN"]
available = pwt["countrycode"].unique()
g7_available = [c for c in g7_codes if c in available]
if len(g7_available) < 5:
    g7_available = list(available[:7])

g7_data = pwt[pwt["countrycode"].isin(g7_available)].copy()
g7_data["log_gdp"] = np.log(g7_data["rgdpna"])

# Panel 2: All 30 countries (heterogeneous -- mix of growth patterns)
mixed_data = pwt.copy()
mixed_data["log_gdp"] = np.log(mixed_data["rgdpna"])

print(
    f"Panel 1 (G7-like): {g7_data['countrycode'].nunique()} countries, "
    f"{g7_data['year'].nunique()} years"
)
print(
    f"Panel 2 (All): {mixed_data['countrycode'].nunique()} countries, "
    f"{mixed_data['year'].nunique()} years"
)

In [ ]:
# Run IPS and LLC on Panel 1 (G7 -- homogeneous)
print("Panel 1: G7-like (Homogeneous)")
print("=" * 60)

results_g7 = {}
try:
    llc_g7 = LLCTest(g7_data, "log_gdp", "countrycode", "year", trend="ct")
    llc_g7_result = llc_g7.run()
    results_g7["LLC"] = {"stat": llc_g7_result.statistic, "pval": llc_g7_result.pvalue}
    print(
        f"LLC: stat={llc_g7_result.statistic:.4f}, p={llc_g7_result.pvalue:.4f} -> {llc_g7_result.conclusion}"
    )
except Exception as e:
    print(f"LLC error: {e}")

try:
    ips_g7 = IPSTest(g7_data, "log_gdp", "countrycode", "year", trend="ct")
    ips_g7_result = ips_g7.run()
    results_g7["IPS"] = {"stat": ips_g7_result.statistic, "pval": ips_g7_result.pvalue}
    print(
        f"IPS: stat={ips_g7_result.statistic:.4f}, p={ips_g7_result.pvalue:.4f} -> {ips_g7_result.conclusion}"
    )
except Exception as e:
    print(f"IPS error: {e}")

In [ ]:
# Run IPS and LLC on Panel 2 (All 30 -- heterogeneous)
print("Panel 2: All 30 Countries (Heterogeneous)")
print("=" * 60)

results_mixed = {}
try:
    llc_mixed = LLCTest(mixed_data, "log_gdp", "countrycode", "year", trend="ct")
    llc_mixed_result = llc_mixed.run()
    results_mixed["LLC"] = {"stat": llc_mixed_result.statistic, "pval": llc_mixed_result.pvalue}
    print(
        f"LLC: stat={llc_mixed_result.statistic:.4f}, p={llc_mixed_result.pvalue:.4f} -> {llc_mixed_result.conclusion}"
    )
except Exception as e:
    print(f"LLC error: {e}")

try:
    ips_mixed = IPSTest(mixed_data, "log_gdp", "countrycode", "year", trend="ct")
    ips_mixed_result = ips_mixed.run()
    results_mixed["IPS"] = {"stat": ips_mixed_result.statistic, "pval": ips_mixed_result.pvalue}
    print(
        f"IPS: stat={ips_mixed_result.statistic:.4f}, p={ips_mixed_result.pvalue:.4f} -> {ips_mixed_result.conclusion}"
    )
except Exception as e:
    print(f"IPS error: {e}")

In [ ]:
# Build comparison table
comparison_rows = []
for panel_name, results in [
    ("G7 (homogeneous)", results_g7),
    ("All 30 (heterogeneous)", results_mixed),
]:
    for test_name, vals in results.items():
        comparison_rows.append(
            {
                "Panel": panel_name,
                "Test": test_name,
                "Statistic": vals["stat"],
                "P-value": vals["pval"],
                "Reject H0 (5%)": vals["pval"] < 0.05,
            }
        )

comp_df = pd.DataFrame(comparison_rows)
print("Comparison: IPS vs LLC across Panel Types")
print("=" * 70)
display(comp_df)
comp_df.to_csv(TABLES_DIR / "01_ex2_ips_vs_llc_comparison.csv", index=False)

### Exercise 2 Discussion

**Key observations:**

1. **Homogeneous panel (G7):** IPS and LLC tend to give similar results because the common-rho assumption of LLC is approximately satisfied. G7 countries have similar growth patterns.

2. **Heterogeneous panel (30 countries):** IPS and LLC may diverge. The mixed panel includes countries with very different growth dynamics (e.g., Chile vs Japan vs Estonia). LLC imposes a single rho for all, which is a poor fit when persistence genuinely varies.

3. **Why they disagree:** LLC assumes $\rho$ is the same for all entities. If some countries have stationary GDP (rare) while most have unit roots, LLC averages toward the unit root and fails to reject. IPS, by allowing heterogeneous $\rho_i$, can detect the stationary subset.

4. **Practical recommendation:** For heterogeneous panels, prefer IPS over LLC. The common-rho assumption is strong and often violated in practice.

---

## Exercise 3: Power Simulation (Hard)

**Task:** Simulate panel data with known rho and measure rejection rates.

In [ ]:
# Step 1: Function to simulate balanced panel AR(1) data
def simulate_panel_ar1(
    N: int, T: int, rho: float, sigma: float = 1.0, seed: int = None
) -> pd.DataFrame:
    """Simulate balanced panel AR(1) data.

    Parameters
    ----------
    N : int
        Number of entities.
    T : int
        Number of time periods.
    rho : float
        AR(1) coefficient.
    sigma : float
        Innovation standard deviation.
    seed : int, optional
        Random seed.

    Returns
    -------
    pd.DataFrame
        Panel data with columns: entity, time, y.
    """
    if seed is not None:
        np.random.seed(seed)
    records = []
    for i in range(N):
        y = np.zeros(T)
        for t in range(1, T):
            y[t] = rho * y[t - 1] + np.random.normal(0, sigma)
        for t in range(T):
            records.append({"entity": i, "time": t, "y": y[t]})
    return pd.DataFrame(records)


# Quick test
test_data = simulate_panel_ar1(5, 20, 0.9, seed=42)
print(f"Simulated data shape: {test_data.shape}")
print(f"Entities: {test_data['entity'].nunique()}, Periods: {test_data['time'].nunique()}")

In [ ]:
# Step 2: Single simulation with rho=0.95
sim_data = simulate_panel_ar1(N=20, T=50, rho=0.95, seed=42)

print("Single simulation: rho=0.95, N=20, T=50")
print("=" * 60)
try:
    ips_sim = IPSTest(sim_data, "y", "entity", "time", trend="c")
    result_sim = ips_sim.run()
    print(f"IPS W-statistic: {result_sim.statistic:.4f}")
    print(f"P-value: {result_sim.pvalue:.4f}")
    print(f"Conclusion: {result_sim.conclusion}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Step 3: Compute rejection rates across rho values
# Using fewer replications for speed (100); increase to 500+ for accuracy
n_replications = 100
N, T = 20, 50
rho_values = [0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.98, 0.99, 1.00]
alpha = 0.05

power_results = []

for rho in rho_values:
    rejections = 0
    errors = 0
    for rep in range(n_replications):
        try:
            sim = simulate_panel_ar1(N, T, rho, seed=rep * 1000 + int(rho * 100))
            ips_test_obj = IPSTest(sim, "y", "entity", "time", trend="c")
            res = ips_test_obj.run()
            if res.pvalue < alpha:
                rejections += 1
        except Exception:
            errors += 1

    valid = n_replications - errors
    rejection_rate = rejections / valid if valid > 0 else np.nan
    power_results.append({"rho": rho, "rejection_rate": rejection_rate, "valid_reps": valid})
    print(f"rho={rho:.2f}: rejection rate = {rejection_rate:.3f} ({valid} valid reps)")

power_df = pd.DataFrame(power_results)
print("\nPower Table:")
display(power_df)
power_df.to_csv(TABLES_DIR / "01_ex3_power_simulation.csv", index=False)

In [ ]:
# Step 4: Plot power curve
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    power_df["rho"],
    power_df["rejection_rate"],
    "o-",
    color="steelblue",
    linewidth=2,
    markersize=8,
    label="IPS test",
)
ax.axhline(0.05, color="red", linestyle="--", alpha=0.7, label="Nominal size (5%)")
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.3)

ax.set_xlabel(r"$\rho$ (AR coefficient)", fontsize=12)
ax.set_ylabel("Rejection Rate", fontsize=12)
ax.set_title(
    f"Power Curve: IPS Test (N={N}, T={T}, {n_replications} replications)",
    fontweight="bold",
    fontsize=13,
)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# Annotate key points
for _, row in power_df.iterrows():
    if row["rho"] in [0.80, 0.95, 1.00]:
        ax.annotate(
            f"{row['rejection_rate']:.2f}",
            (row["rho"], row["rejection_rate"]),
            textcoords="offset points",
            xytext=(10, 10),
            fontsize=10,
            fontweight="bold",
        )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_ex3_power_curve.png", dpi=300, bbox_inches="tight")
plt.show()

print("Key observations:")
print(
    f"  rho=1.00 (H0 true): rejection rate should be near {alpha:.2f} (actual: {power_df[power_df['rho'] == 1.0]['rejection_rate'].values[0]:.3f})"
)
print(
    f"  rho=0.95: rejection rate = {power_df[power_df['rho'] == 0.95]['rejection_rate'].values[0]:.3f}"
)
print(
    f"  rho=0.80: rejection rate = {power_df[power_df['rho'] == 0.80]['rejection_rate'].values[0]:.3f}"
)

### Exercise 3 Discussion

**Results:**

1. **Size (rho = 1.0):** The rejection rate at the true null should be close to 5% (the nominal level). If it's substantially higher, the test is oversized; if lower, it's conservative.

2. **Power curve shape:** Power increases as rho moves away from 1.0. The further rho is from unity, the easier it is to detect that the series is stationary.

3. **rho = 0.95 power:** With N=20 and T=50, the IPS test has moderate power against near-unit-root alternatives. This is the regime where panel tests have a clear advantage over univariate ADF tests.

4. **Practical implications:**
   - Very persistent series (rho > 0.97) are difficult to distinguish from unit roots
   - Increasing N (more entities) improves power significantly
   - Increasing T helps but with diminishing returns
   - The panel dimension is crucial for power in finite samples

In [ ]:
# Bonus: Effect of N and T on power at rho=0.95
print("Effect of N and T on Power (rho=0.95, 50 replications)")
print("=" * 60)

n_reps_bonus = 50  # fewer reps for speed
rho_fixed = 0.95

nt_combos = [(10, 30), (10, 50), (20, 30), (20, 50), (50, 30), (50, 50)]
nt_results = []

for N_val, T_val in nt_combos:
    rejections = 0
    valid = 0
    for rep in range(n_reps_bonus):
        try:
            sim = simulate_panel_ar1(N_val, T_val, rho_fixed, seed=rep * 7 + N_val + T_val)
            ips_obj = IPSTest(sim, "y", "entity", "time", trend="c")
            res = ips_obj.run()
            valid += 1
            if res.pvalue < 0.05:
                rejections += 1
        except Exception:
            pass
    rate = rejections / valid if valid > 0 else np.nan
    nt_results.append({"N": N_val, "T": T_val, "Power": rate})
    print(f"  N={N_val:3d}, T={T_val:3d}: power = {rate:.3f}")

nt_df = pd.DataFrame(nt_results)
print("\nPower increases with both N and T.")
print("N has a particularly strong effect because IPS averages across N entity-specific tests.")

---

*End of solutions for Notebook 01: Panel Unit Root Tests*